### Notebook for extracting and analysing LFP data from Neuropixels recordings

#### Imports

In [ ]:
# Imports
import os
import yaml

import ephyviewer
import numpy as np
import matplotlib.pyplot as plt
import spikeinterface.full as si
import spikeinterface.extractors as se
import spikeinterface.preprocessing as sp

from scipy.signal import spectrogram
from probeinterface import get_probe, read_probeinterface, Probe
from probeinterface.plotting import plot_probe
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

#### Load configs and set up recording object

In [ ]:
load_dotenv()
repo_root = Path(find_dotenv()).parent

raw_data_dir_env = "DUAL_RAW_DATA_DIR"
data_fname_env = "AWAKE_TEST_DATA_FNAME_PL"
probe_config_env = "PROBE_CONFIG_PATH_PL"
derivatives_dir_env = "DERIVATIVES_DIR"

raw_data_dir = Path(os.getenv(raw_data_dir_env))
data_fname = Path(os.getenv(data_fname_env))
probe_config_path = Path(os.getenv(probe_config_env))
derivatives_dir = Path(os.getenv(derivatives_dir_env))
full_data_path = os.path.join(raw_data_dir, data_fname)
config_path = os.path.join(repo_root, "config/preprocessing.yml")
lfp_output_dir = os.path.join(derivatives_dir, "lfp")

# Load preprocessing config
with open(Path(config_path)) as f:
    config = yaml.safe_load(f)
    
sample_rate = config['recording']['sample_rate']
n_channels = config['recording']['n_channels']
dtype = config['recording']['dtype']
freq_min = config['lfp']['bandpass_filter']['freq_min']
freq_max = config['lfp']['bandpass_filter']['freq_max']
reference = config['lfp']['common_reference']['reference']
operator = config['lfp']['common_reference']['operator']
resample_rate = config['lfp']['resample_rate']

In [ ]:
recording = se.read_binary(full_data_path, sample_rate, dtype, n_channels)

#### Interactive viewer for frequency band inspection

In [ ]:
recording_ch0 = recording.select_channels([recording.channel_ids[0]])

sig_source = ephyviewer.SpikeInterfaceRecordingSource(recording=recording_ch0)

# Low frequency (< 1 Hz)
filtered_recording = si.bandpass_filter(recording_ch0, freq_min=0.5, freq_max=1.)
sig_filtered_source_low = ephyviewer.SpikeInterfaceRecordingSource(recording=filtered_recording)

# Delta (1-4 Hz)
filtered_recording = si.bandpass_filter(recording_ch0, freq_min=1., freq_max=4.)
sig_filtered_source_delta = ephyviewer.SpikeInterfaceRecordingSource(recording=filtered_recording)

# Theta (4-8 Hz)
filtered_recording = si.bandpass_filter(recording_ch0, freq_min=4., freq_max=8.)
sig_filtered_source_theta = ephyviewer.SpikeInterfaceRecordingSource(recording=filtered_recording)

# Alpha (8-12 Hz)
filtered_recording = si.bandpass_filter(recording_ch0, freq_min=8., freq_max=12.)
sig_filtered_source_alpha = ephyviewer.SpikeInterfaceRecordingSource(recording=filtered_recording)

# Beta (12-30 Hz)
filtered_recording = si.bandpass_filter(recording_ch0, freq_min=12., freq_max=30.)
sig_filtered_source_beta = ephyviewer.SpikeInterfaceRecordingSource(recording=filtered_recording)

# Gamma (30-100 Hz)
filtered_recording = si.bandpass_filter(recording_ch0, freq_min=30., freq_max=100.)
sig_filtered_source_gamma = ephyviewer.SpikeInterfaceRecordingSource(recording=filtered_recording)

# Ripple (100-180 Hz)
filtered_recording = si.bandpass_filter(recording_ch0, freq_min=100., freq_max=180.)
sig_filtered_source_ripple = ephyviewer.SpikeInterfaceRecordingSource(recording=filtered_recording)

app = ephyviewer.mkQApp()
win = ephyviewer.MainViewer(debug=True, show_auto_scale=True)

view = ephyviewer.TraceViewer(source=sig_source, name='signals')
win.add_view(view)

view = ephyviewer.TraceViewer(source=sig_filtered_source_low, name='signals filtered - Low')
win.add_view(view)

view = ephyviewer.TraceViewer(source=sig_filtered_source_delta, name='signals filtered - Delta')
win.add_view(view)

view = ephyviewer.TraceViewer(source=sig_filtered_source_theta, name='signals filtered - Theta')
win.add_view(view)

view = ephyviewer.TraceViewer(source=sig_filtered_source_alpha, name='signals filtered - Alpha')
win.add_view(view)

view = ephyviewer.TraceViewer(source=sig_filtered_source_beta, name='signals filtered - Beta')
win.add_view(view)

view = ephyviewer.TraceViewer(source=sig_filtered_source_gamma, name='signals filtered - Gamma')
win.add_view(view)

view = ephyviewer.TraceViewer(source=sig_filtered_source_ripple, name='signals filtered - Ripple')
win.add_view(view)

win.show()
app.exec()

#### Read probe layout from config

In [ ]:
probe_group = read_probeinterface(probe_config_path)
probe = probe_group.probes[0]

active_mask = probe.device_channel_indices != -1
active_probe = probe.get_slice(active_mask)

fig, ax = plt.subplots(figsize=(4, 12))
colors = ["blue" if ch != -1 else "white" for ch in probe.device_channel_indices]
plot_probe(probe, ax=ax, contacts_colors=colors, with_device_index=False)
plt.tight_layout()

recording_with_probe = recording.set_probe(probe)

In [ ]:
lfp = sp.bandpass_filter(recording_with_probe, freq_min=0.5, freq_max=300)
lfp = sp.common_reference(lfp, reference="global", operator="median")
lfp = sp.resample(lfp, resample_rate=1250)

In [ ]:
fs = lfp.get_sampling_frequency()  # 1250
chunk_size = int(60 * fs)  # 5 minutes at a time
n_total = lfp.get_num_samples()
channel_idx = 6

all_freqs, all_times, all_Sxx = [], [], []

for (loop_no, start) in enumerate(range(0, chunk_size*5, chunk_size)):
    end = min(start + chunk_size, n_total)
    chunk = lfp.get_traces(start_frame=start, end_frame=end)
    lfp_trace = chunk[:, channel_idx]
    freqs, times, Sxx = spectrogram(
        lfp_trace,
        fs=1250,
        window='hann',
        nperseg=1250 * 4,    # 4-second window
        noverlap=1250 * 3,   # 75% overlap → 1-second steps
    )
    all_freqs.append(freqs)
    all_times.append(times)
    all_Sxx.append(Sxx)
    print(f'Finished chunk {loop_no}')

In [ ]:
freqs = all_freqs[0]  # same for all chunks

# Offset times by chunk start, then concatenate
times = np.concatenate([
    all_times[i] + (i * 60)  # offset by chunk number * chunk duration in seconds
    for i in range(len(all_times))
])

Sxx = np.concatenate(all_Sxx, axis=1)  # concatenate along time axis

print(f"freqs shape: {freqs.shape}")
print(f"times shape: {times.shape}")
print(f"Sxx shape:   {Sxx.shape}")  # should be (n_freqs, n_times)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 5))

ax.pcolormesh(
    times,           # convert to hours
    freqs,
    10 * np.log10(Sxx),     # convert to dB
    cmap="inferno",
    shading="auto",
    vmin=-10,
    vmax=40
)

ax.set_ylim(0, 30)          # focus on sleep-relevant bands
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title(f"LFP spectrogram — channel {channel_idx}")

plt.colorbar(ax.collections[0], ax=ax, label="Power (dB)")
plt.tight_layout()
plt.show()

In [ ]:
probe_group = read_probeinterface(Path(os.getenv("PROBE_CONFIG_PATH_PL")))
probe = probe_group.probes[0]

fig, ax = plt.subplots(figsize=(4, 12))
colors = ["blue" if ch != -1 else "white" for ch in probe.device_channel_indices]
plot_probe(probe, ax=ax, contacts_colors=colors, with_device_index=False)
plt.tight_layout()